# Customer Churn Prediction
Works with **any** churn dataset — just set `TARGET_COL` and `DROP_COLS` to match your data.

**Steps:**
1. Install packages
2. Load your data
3. Explore the data
4. Train the model
5. Predict on new customers

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn --quiet
print('Ready.')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from churn.churn_model import ChurnModel
print('Imports done.')

## 1. Load Data
Swap the sample data below with your own CSV.
Change `TARGET_COL` to your churn label column name.

In [ ]:
# --- CHANGE THESE ---
TARGET_COL = 'churned'
DROP_COLS  = ['customer_id']
# --------------------

# Replace with: df = pd.read_csv('your_file.csv')
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'customer_id':      range(n),
    'tenure_months':    np.random.randint(1, 72, n),
    'monthly_charges':  np.round(np.random.uniform(20, 120, n), 2),
    'total_charges':    np.round(np.random.uniform(100, 8000, n), 2),
    'num_products':     np.random.randint(1, 6, n),
    'support_calls':    np.random.randint(0, 10, n),
    'contract_type':    np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n),
    'payment_method':   np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], n),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n),
    'churned':          np.random.choice([0, 1], n, p=[0.73, 0.27])
})

print(f'Rows: {len(df)} | Churn rate: {df[TARGET_COL].mean():.1%}')
df.head()

## 2. Explore the Data

In [ ]:
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Churn breakdown:\n{df[TARGET_COL].value_counts()}')

In [ ]:
# Numeric feature distributions by churn
num_cols = [c for c in df.select_dtypes(include=['int64','float64']).columns
            if c not in [TARGET_COL] + DROP_COLS]

fig, axes = plt.subplots(1, len(num_cols), figsize=(5*len(num_cols), 4))
if len(num_cols) == 1: axes = [axes]
for ax, col in zip(axes, num_cols):
    for label, grp in df.groupby(TARGET_COL):
        grp[col].plot(kind='kde', ax=ax, label=f'Churn={label}')
    ax.set_title(col); ax.legend()
plt.suptitle('Feature Distributions by Churn', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Churn rate by categorical feature
cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in DROP_COLS]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5*len(cat_cols), 4))
if len(cat_cols) == 1: axes = [axes]
for ax, col in zip(axes, cat_cols):
    df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False).plot(
        kind='bar', ax=ax, color='#e74c3c', alpha=0.8)
    ax.axhline(df[TARGET_COL].mean(), linestyle='--', color='black', lw=1, label='Average')
    ax.set_title(f'{col}'); ax.legend()
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.suptitle('Churn Rate by Category', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 3. Train the Model
Tests all 3 algorithms, tunes each one, picks the best.

In [ ]:
model = ChurnModel(target_col=TARGET_COL, drop_cols=DROP_COLS)
model.fit(df)

## 4. Predict on New Customers

In [ ]:
new_customers = pd.DataFrame({
    'customer_id':      [9001, 9002, 9003],
    'tenure_months':    [2,    36,   60  ],
    'monthly_charges':  [95.0, 45.0, 30.0],
    'total_charges':    [190.0,1620.0,1800.0],
    'num_products':     [1, 3, 5],
    'support_calls':    [8, 1, 0],
    'contract_type':    ['Month-to-Month', 'One Year', 'Two Year'],
    'payment_method':   ['Electronic Check', 'Credit Card', 'Bank Transfer'],
    'internet_service': ['Fiber', 'DSL', 'DSL']
})

results = model.predict(new_customers)
pd.concat([new_customers[['customer_id','tenure_months','contract_type']], results], axis=1)